#### Silver Transformation

Purpose

- Read latest Bronze Delta data
- Parse raw JSON
- Flatten nested structures
- Apply data quality rules
- Create Silver dataset
- Create Quarantine dataset

Output

Silver:
/Volumes/assignment/university/silver/university_chapters

Quarantine:
/Volumes/assignment/university/quarantine/university_chapters/[run_id]

In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *
from datetime import datetime

In [0]:
bronze_root = "/Volumes/assignment/university/bronze/university_chapters"

In [0]:
folders = dbutils.fs.ls(bronze_root)

run_folders = [f.path for f in folders if f.isDir()]

latest_run = sorted(run_folders)[-1]

# Extract run_id
run_id = latest_run.rstrip("/").split("/")[-1]

print(f"Latest Run Path : {latest_run}")
print(f"Run ID          : {run_id}")

Latest Run Path : dbfs:/Volumes/assignment/university/bronze/university_chapters/20260722_031302_11b5b2b6/
Run ID          : 20260722_031302_11b5b2b6


In [0]:
bronze_df = (
    spark.read
         .format("json")
         .load(latest_run)
)

In [0]:
display(bronze_df)

raw_json
"{""attributes"": {""OBJECTID"": 19, ""University_Chapter"": ""California Polytechnic State University"", ""City"": ""San Luis Obispo"", ""State"": ""CA"", ""ChapterID"": ""CA-0355"", ""MEVR_RD"": ""Derek Swindall""}, ""geometry"": {""x"": -120.66319100299995, ""y"": 35.274309145000075}}"
"{""attributes"": {""OBJECTID"": 26, ""University_Chapter"": ""Chico State University"", ""City"": ""Chico"", ""State"": ""CA"", ""ChapterID"": ""CA-0300"", ""MEVR_RD"": ""Brian Henderson""}, ""geometry"": {""x"": -121.83546324499997, ""y"": 39.73998176200007}}"
"{""attributes"": {""OBJECTID"": 1757, ""University_Chapter"": ""Fresno State"", ""City"": ""Fresno"", ""State"": ""CA"", ""ChapterID"": ""CA-0362"", ""MEVR_RD"": ""Garrett Williams""}, ""geometry"": {""x"": -119.73959481924653, ""y"": 36.82354541564933}}"


In [0]:
sample_json = bronze_df.select("raw_json").first()[0]

json_schema = schema_of_json(lit(sample_json))

In [0]:
parsed_df = bronze_df.withColumn(
    "json_data",
    from_json(col("raw_json"), json_schema)
)

In [0]:
display(parsed_df)

raw_json,json_data
"{""attributes"": {""OBJECTID"": 19, ""University_Chapter"": ""California Polytechnic State University"", ""City"": ""San Luis Obispo"", ""State"": ""CA"", ""ChapterID"": ""CA-0355"", ""MEVR_RD"": ""Derek Swindall""}, ""geometry"": {""x"": -120.66319100299995, ""y"": 35.274309145000075}}","List(List(CA-0355, San Luis Obispo, Derek Swindall, 19, CA, California Polytechnic State University), List(-120.66319100299995, 35.274309145000075))"
"{""attributes"": {""OBJECTID"": 26, ""University_Chapter"": ""Chico State University"", ""City"": ""Chico"", ""State"": ""CA"", ""ChapterID"": ""CA-0300"", ""MEVR_RD"": ""Brian Henderson""}, ""geometry"": {""x"": -121.83546324499997, ""y"": 39.73998176200007}}","List(List(CA-0300, Chico, Brian Henderson, 26, CA, Chico State University), List(-121.83546324499997, 39.73998176200007))"
"{""attributes"": {""OBJECTID"": 1757, ""University_Chapter"": ""Fresno State"", ""City"": ""Fresno"", ""State"": ""CA"", ""ChapterID"": ""CA-0362"", ""MEVR_RD"": ""Garrett Williams""}, ""geometry"": {""x"": -119.73959481924653, ""y"": 36.82354541564933}}","List(List(CA-0362, Fresno, Garrett Williams, 1757, CA, Fresno State), List(-119.73959481924653, 36.82354541564933))"


In [0]:
silver_df = parsed_df.select(
    col("json_data.attributes.OBJECTID").alias("source_object_id"),
    col("json_data.attributes.ChapterID").alias("chapter_id"),
    col("json_data.attributes.University_Chapter").alias("chapter_name"),
    col("json_data.attributes.City").alias("city"),
    col("json_data.attributes.State").alias("state"),
    col("json_data.geometry.x").alias("longitude"),
    col("json_data.geometry.y").alias("latitude"),
    concat_ws(
        ",",
        round(col("json_data.geometry.x"),6).cast("string"),
        round(col("json_data.geometry.y"),6).cast("string"),
    ).alias("longitude/latitude")
)

In [0]:
display(silver_df)

source_object_id,chapter_id,chapter_name,city,state,longitude,latitude,longitude/latitude
19,CA-0355,California Polytechnic State University,San Luis Obispo,CA,-120.66319100299995,35.274309145000075,"-120.663191,35.274309"
26,CA-0300,Chico State University,Chico,CA,-121.83546324499997,39.73998176200007,"-121.835463,39.739982"
1757,CA-0362,Fresno State,Fresno,CA,-119.73959481924653,36.82354541564933,"-119.739595,36.823545"


In [0]:
silver_df = silver_df.dropDuplicates(["chapter_id"])

In [0]:
display(silver_df)

source_object_id,chapter_id,chapter_name,city,state,longitude,latitude,longitude/latitude
19,CA-0355,California Polytechnic State University,San Luis Obispo,CA,-120.66319100299995,35.274309145000075,"-120.663191,35.274309"
26,CA-0300,Chico State University,Chico,CA,-121.83546324499997,39.73998176200007,"-121.835463,39.739982"
1757,CA-0362,Fresno State,Fresno,CA,-119.73959481924653,36.82354541564933,"-119.739595,36.823545"


In [0]:
valid_coordinates = (
    col("longitude").between(-180,180)
    &
    col("latitude").between(-90,90)
)

In [0]:
quarantine_df = silver_df.filter(~valid_coordinates)

In [0]:
display(quarantine_df)

source_object_id,chapter_id,chapter_name,city,state,longitude,latitude,longitude/latitude


In [0]:
quarantine_df = quarantine_df.withColumn(
    "reason_code",
    lit("INVALID_COORDINATES")
)

In [0]:
silver_df = silver_df.filter(valid_coordinates)

In [0]:
silver_df = silver_df.withColumn(
    "dq_status",
    when(
        col("city").isNull() | (trim(col("city")) == "") | (upper(col("city")) == "UNKNOWN"),
        "WARNING"
    ).otherwise("OK")
)

In [0]:
display(silver_df)

source_object_id,chapter_id,chapter_name,city,state,longitude,latitude,longitude/latitude,dq_status
19,CA-0355,California Polytechnic State University,San Luis Obispo,CA,-120.66319100299995,35.274309145000075,"-120.663191,35.274309",OK
26,CA-0300,Chico State University,Chico,CA,-121.83546324499997,39.73998176200007,"-121.835463,39.739982",OK
1757,CA-0362,Fresno State,Fresno,CA,-119.73959481924653,36.82354541564933,"-119.739595,36.823545",OK


In [0]:
silver_df = silver_df.withColumn(
    "dq_warnings",
    when(
        col("dq_status")=="WARNING",
        array(lit("MISSING_OR_UNKNOWN_CITY"))
    ).otherwise(array())
)

In [0]:
display(silver_df)

source_object_id,chapter_id,chapter_name,city,state,longitude,latitude,longitude/latitude,dq_status,dq_warnings
19,CA-0355,California Polytechnic State University,San Luis Obispo,CA,-120.66319100299995,35.274309145000075,"-120.663191,35.274309",OK,List()
26,CA-0300,Chico State University,Chico,CA,-121.83546324499997,39.73998176200007,"-121.835463,39.739982",OK,List()
1757,CA-0362,Fresno State,Fresno,CA,-119.73959481924653,36.82354541564933,"-119.739595,36.823545",OK,List()


In [0]:
silver_df = (
    silver_df
        .withColumn("processed_timestamp", current_timestamp())
        .withColumn("source_run_folder", lit(latest_run))
)

In [0]:
display(silver_df)

source_object_id,chapter_id,chapter_name,city,state,longitude,latitude,longitude/latitude,dq_status,dq_warnings,processed_timestamp,source_run_folder
19,CA-0355,California Polytechnic State University,San Luis Obispo,CA,-120.66319100299995,35.274309145000075,"-120.663191,35.274309",OK,List(),2026-07-22T05:15:55.292Z,dbfs:/Volumes/assignment/university/bronze/university_chapters/20260722_031302_11b5b2b6/
26,CA-0300,Chico State University,Chico,CA,-121.83546324499997,39.73998176200007,"-121.835463,39.739982",OK,List(),2026-07-22T05:15:55.292Z,dbfs:/Volumes/assignment/university/bronze/university_chapters/20260722_031302_11b5b2b6/
1757,CA-0362,Fresno State,Fresno,CA,-119.73959481924653,36.82354541564933,"-119.739595,36.823545",OK,List(),2026-07-22T05:15:55.292Z,dbfs:/Volumes/assignment/university/bronze/university_chapters/20260722_031302_11b5b2b6/


In [0]:
silver_path = "/Volumes/assignment/university/silver/university_chapters"

(
    silver_df.write
        .format("delta")
        .mode("overwrite")
        .save(silver_path)
)

In [0]:
quarantine_path = "/Volumes/assignment/university/quarantine/university_chapters/{run_id}"

(
    quarantine_df.write
        .format("delta")
        .mode("overwrite")
        .saveAsTable('quarantine_table')
)

In [0]:
print("=" * 60)
print(f"Run Folder      : {latest_run}")
print(f"Bronze Records  : {bronze_df.count()}")
print(f"Silver Records  : {silver_df.count()}")
print(f"Quarantine Rows : {quarantine_df.count()}")
print("=" * 60)

Run Folder      : dbfs:/Volumes/assignment/university/bronze/university_chapters/20260722_031302_11b5b2b6/
Bronze Records  : 3
Silver Records  : 3
Quarantine Rows : 0
